# LAX — CPU-only Google Colab launcher

This notebook launches the real Streamlit application from the GitHub repository using **CPU mode**. It does not use a GPU, retrain the adapter, or modify saved adapter weights.

**Note:** CPU generation is slower. The psychology model is loaded in FP32, and the second general-purpose language model is disabled by default in CPU mode to reduce RAM use. LAX keeps its grounded psychology flow and uses lightweight replies for greetings and simple general chat.

## 1. Connect Google Drive and find the saved adapter

The expected folder is `MyDrive/psychology-qwen-qlora-v2`. If Drive authentication fails, the cell continues and the next cell allows you to upload a ZIP of that adapter folder.

In [ ]:
from pathlib import Path

ADAPTER_PATH = None
DRIVE_ADAPTER = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    if (DRIVE_ADAPTER / "adapter_model.safetensors").is_file():
        ADAPTER_PATH = DRIVE_ADAPTER
        print("Found adapter in Google Drive:", ADAPTER_PATH)
    else:
        print("Drive mounted, but adapter weights were not found at:", DRIVE_ADAPTER)
except Exception as exc:
    print("Google Drive mount did not succeed:", repr(exc))
    print("You can continue by uploading a ZIP of the adapter folder in the next cell.")

print("Adapter ready:", ADAPTER_PATH is not None)

## 2. Upload the adapter ZIP only if Drive was unavailable

Skip this cell if the previous cell printed `Adapter ready: True`. If needed, first download the `psychology-qwen-qlora-v2` folder as a ZIP from Google Drive on your computer.

In [ ]:
if ADAPTER_PATH is None:
    from google.colab import files
    import zipfile

    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError(
            "Upload a ZIP containing adapter_model.safetensors and adapter_config.json."
        )

    extract_root = Path("/content/lax-adapter-upload")
    extract_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_names[0]) as archive:
        # Refuse unsafe archive paths before extraction.
        root = extract_root.resolve()
        for member in archive.infolist():
            target = (extract_root / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError("Unsafe path inside uploaded ZIP.")
        archive.extractall(extract_root)

    weight_files = list(extract_root.rglob("adapter_model.safetensors"))
    if len(weight_files) != 1:
        raise FileNotFoundError(
            f"Expected one adapter_model.safetensors, found {len(weight_files)}."
        )

    ADAPTER_PATH = weight_files[0].parent

if not (ADAPTER_PATH / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Adapter weights are missing: {ADAPTER_PATH}")
if not (ADAPTER_PATH / "adapter_config.json").is_file():
    raise FileNotFoundError(f"adapter_config.json is missing: {ADAPTER_PATH}")

print("Adapter path:", ADAPTER_PATH)
print("Adapter weights and configuration verified.")

## 3. Clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/psychology-raft-rag-chatbot")
REPO_URL = "https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git"

if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "app.py").is_file(), "app.py was not found in the repository."
assert (REPO_DIR / "requirements-colab-cpu.txt").is_file(), "CPU requirements file is missing."
print("Repository is ready:", REPO_DIR)

## 5a. Fix an old torchaudio package if one is preinstalled

Some Colab runtimes have an old `torchaudio 0.10.0`, which can stop Transformers from starting. This cell checks the installed version and, only when it is older than 0.16, installs the CPU torchaudio version matching the current PyTorch version. It does not install CUDA/GPU packages or replace PyTorch.

In [ ]:
import importlib.metadata as metadata
from packaging.version import Version
import subprocess
import sys
import torch

try:
    torchaudio_version = metadata.version("torchaudio")
except metadata.PackageNotFoundError:
    torchaudio_version = None

torch_version = torch.__version__.split("+")[0]
print("PyTorch:", torch.__version__)
print("Existing torchaudio:", torchaudio_version)

if torchaudio_version is not None and Version(torchaudio_version) <= Version("0.16.0"):
    print(f"Replacing incompatible torchaudio {torchaudio_version} with {torch_version} (CPU wheel).")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-deps",
            "--index-url", "https://download.pytorch.org/whl/cpu",
            f"torchaudio=={torch_version}",
        ],
        check=True,
    )
    print("Restart the Colab runtime if Python reports that torchaudio is already loaded.")
else:
    print("torchaudio version does not need this compatibility repair.")

try:
    print("torchaudio after check:", metadata.version("torchaudio"))
except metadata.PackageNotFoundError:
    print("torchaudio is not installed; LAX does not directly use audio input.")

## 4. Force CPU mode before importing the app

This hides any available GPU from this notebook and sets the adapter path for Streamlit. No CUDA package is installed or needed for this CPU launcher.

In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["LAX_ADAPTER_PATH"] = str(ADAPTER_PATH)
os.environ["LAX_LOAD_GENERAL_ON_CPU"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%cd /content/psychology-raft-rag-chatbot
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Adapter:", os.environ["LAX_ADAPTER_PATH"])
assert not torch.cuda.is_available(), "CUDA is visible. Restart with a CPU runtime and run again."

## 5. Install the CPU-only app dependencies

The requirements file deliberately does not reinstall PyTorch or bitsandbytes. This helps preserve the CPU PyTorch build already supplied by Colab.

In [ ]:
%pip install -q -r /content/psychology-raft-rag-chatbot/requirements-colab-cpu.txt

In [ ]:
import importlib.metadata as metadata
import torch

for package in [
    "transformers", "peft", "accelerate",
    "sentence-transformers", "faiss-cpu", "streamlit", "safetensors"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"MISSING: {package}")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 6. Build the local knowledge index if it does not exist

This uses the repository's included markdown knowledge files. It does not touch or retrain your adapter.

In [ ]:
import subprocess
index_dir = REPO_DIR / "data" / "vector_index"
index_file = index_dir / "student_wellbeing.faiss"
metadata_file = index_dir / "metadata.json"

if not (index_file.is_file() and metadata_file.is_file()):
    subprocess.run(
        [
            sys.executable, "inference/rag_local.py",
            "--source-dir", "data/sources",
            "--index-dir", "data/vector_index",
        ],
        cwd=str(REPO_DIR),
        check=True,
    )
else:
    print("Existing local FAISS index found; keeping it.")

print("Index ready:", index_file.is_file() and metadata_file.is_file())

## 7. Start Streamlit and inspect its log

In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path("/content/lax.log")
log_handle = log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Streamlit PID:", streamlit_process.pid)
print("Waiting for the local server (the first model load can take time)...")

ready = False
for attempt in range(45):
    time.sleep(2)
    if streamlit_process.poll() is not None:
        print("Streamlit exited with code:", streamlit_process.returncode)
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass

print("Local Streamlit health check passed:", ready)
print("\n--- Recent LAX log output ---")
print(log_path.read_text(errors="replace")[-10000:])

if not ready:
    raise RuntimeError("Streamlit did not become ready. Read the log above before retrying.")

## 8. Create a temporary public link using Cloudflare Tunnel

Run this cell last. It stays active while the link is being served. Copy the `https://….trycloudflare.com` URL printed in the output. Do not close or interrupt this cell while using LAX.

In [ ]:
import re
import shutil
import subprocess
import time
from pathlib import Path
import urllib.request

# Install cloudflared automatically if the earlier install cell was skipped
# or failed. Uses the Debian package and lets apt resolve dependencies.
cloudflared = shutil.which("cloudflared")
if cloudflared is None:
    deb_path = Path("/content/cloudflared-linux-amd64.deb")
    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64.deb"
    )

    print("cloudflared is missing. Downloading the official package...")
    subprocess.run(
        ["wget", "-q", "--show-progress", download_url, "-O", str(deb_path)],
        check=True,
    )

    if not deb_path.is_file() or deb_path.stat().st_size < 1_000_000:
        raise RuntimeError(
            "The cloudflared package download appears incomplete. "
            "Retry this cell or inspect network access."
        )

    print("Installing cloudflared and resolving package dependencies...")
    subprocess.run(
        ["apt-get", "install", "-y", str(deb_path)],
        check=True,
    )
    cloudflared = shutil.which("cloudflared")

if cloudflared is None:
    raise RuntimeError(
        "cloudflared still was not found after installation. "
        "Run !dpkg -l cloudflared and share the output."
    )

version = subprocess.run(
    [cloudflared, "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print("Installed:", version.stdout.strip())

# Confirm Streamlit is answering locally before opening the public tunnel.
try:
    with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError(f"Streamlit health check returned HTTP {response.status}")
except Exception as exc:
    raise RuntimeError(
        "Streamlit is not healthy at http://127.0.0.1:8501. "
        "Run the Streamlit startup cell and inspect /content/lax.log before starting the tunnel."
    ) from exc

log_path = Path("/content/cloudflared.log")
log_path.write_text("", encoding="utf-8")
log_handle = log_path.open("a", encoding="utf-8")

cloudflared_process = subprocess.Popen(
    [
        cloudflared, "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--protocol", "http2",
        "--loglevel", "info",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

print("Cloudflare Tunnel started in the background.")
print("Process ID:", cloudflared_process.pid)
print("Checking for a public URL (up to 120 seconds)...")

public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    if cloudflared_process.poll() is not None:
        break

    logs = log_path.read_text(encoding="utf-8", errors="replace")
    found = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if found:
        public_url = found[-1]
        break
    time.sleep(2)

if public_url:
    print()\n    print("LAX public URL:", public_url)
    print("Open the URL in a new browser tab.")
    print("Keep this Colab runtime active while using LAX.")
else:
    logs = log_path.read_text(encoding="utf-8", errors="replace")
    print("No public URL appeared within 120 seconds.")
    print()\n    print("--- Last Cloudflare logs ---")
    print(logs[-12000:] if logs else "(Cloudflare log is empty)")
    if cloudflared_process.poll() is not None:
        print("Cloudflare exited with code:", cloudflared_process.returncode)
    print()\n    print("If logs show connection refused to 127.0.0.1:8501, restart Streamlit first.")
    print("For DNS or network errors, retry on a working network.")

## Troubleshooting

- **Adapter not found:** confirm the Drive folder contains both `adapter_model.safetensors` and `adapter_config.json`, or upload a ZIP containing them.
- **Drive credential error:** use the ZIP-upload option in Cell 2.
- **Host error:** inspect `/content/lax.log` in Colab; fix the local Streamlit/backend error before restarting the tunnel.
- **Out of memory:** restart the runtime and rerun from the top. CPU mode does not load the secondary Qwen3 general chat model by default, but the fine-tuned Qwen2.5 model still needs substantial RAM.
- **Slow replies:** CPU inference may take a long time for a 1.5B model. This notebook does not use GPU quota.

LAX is a student wellbeing psychoeducation assistant, not a therapist or emergency service.